# Syria Hydrography CRS Alignment Map

Overview

Aligns river and mapped water-body datasets that originally use different coordinate reference systems.
River features are reprojected from Web Mercator (EPSG:3857) to WGS 84 (EPSG:4326) so that both hydrography datasets can be stored, compared and displayed in the same spatial reference.
The full-resolution aligned datasets are saved in a GeoPackage, while simplified copies are used for the interactive map.

異なる座標参照系を使用している河川データと水域データを、同じ座標参照系へ統一します。
河川データをWeb Mercator（EPSG:3857）からWGS 84（EPSG:4326）へ変換し、2つの水文データを同じ空間基準で保存、比較および表示できるようにします。
CRS統一後の全解像度データはGeoPackageへ保存し、インタラクティブ地図には軽量化したコピーを使用します。

Objectives

- Read and validate the river, water-body and administrative boundary datasets
- Confirm the source coordinate reference systems
- Reproject the river dataset from EPSG:3857 to EPSG:4326
- Verify that reprojection preserves the river features and geometry types
- Save the aligned river and water-body datasets in a GeoPackage
- Create simplified copies for web-map display
- Visualise the aligned hydrography datasets in Folium
- Create an interactive map with labels, information panels, a legend and layer controls

- 河川、水域および行政界データを読み込み、検証する
- 各入力データの座標参照系を確認する
- 河川データをEPSG:3857からEPSG:4326へ変換する
- CRS変換後も河川の地物数とジオメトリ形式が保持されていることを確認する
- CRS統一後の河川データと水域データをGeoPackageへ保存する
- Web地図表示用の軽量化コピーを作成する
- CRSを統一した水文データをFoliumで可視化する
- ラベル、情報パネル、凡例およびレイヤー切り替え機能を備えたインタラクティブ地図を作成する

Workflow

#### English

1. Define the input and output file paths
2. Read the hydrography and administrative boundary datasets
3. Validate the coordinate reference systems, attributes and geometries
4. Reproject the river dataset to EPSG:4326
5. Validate the reprojected river dataset
6. Save the aligned hydrography datasets in a GeoPackage
7. Read back and verify the saved GeoPackage layers
8. Create simplified copies for web-map display
9. Add the hydrography, administrative boundary and label layers
10. Add the information panel, legend and layer controls
11. Save and display the interactive map

#### 日本語

1. 入出力ファイルのパスを設定する
2. 水文データと行政界データを読み込む
3. 座標参照系、属性およびジオメトリを検証する
4. 河川データをEPSG:4326へ変換する
5. CRS変換後の河川データを検証する
6. CRSを統一した水文データをGeoPackageへ保存する
7. 保存したGeoPackageを再読込して確認する
8. Web地図表示用の軽量化コピーを作成する
9. 水文、行政界およびラベルの各レイヤーを追加する
10. 情報パネル、凡例およびレイヤー切り替え機能を追加する
11. インタラクティブ地図を保存し、Notebook上に表示する

Data

Hydrography data:

- `syr_rivers_3857.geojson`
- `syr_lakes_4326.geojson`

Source: OpenStreetMap contributors

Administrative boundary data:

- `syr_admin0.geojson`
- `syr_admin1.geojson`

Source: HDX OCHA, Syria subnational administrative boundaries

Technologies

- Python
- GeoPandas
- Folium
- GeoPackage

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

from pathlib import Path

import folium
import geopandas as gpd
from branca.element import Element

In [ ]:
# 2
# Define the input and output file paths
# 入力データと出力ファイルのパスを設定する

PROJECT_DIR = Path(
    "/Users/marisa/Syria_Humanitarian_Climate_Facts/" "01_PROJECTS/03_VECTOR_OPERATIONS"
)

DATA_DIR = Path("/Users/marisa/Syria_Humanitarian_Climate_Facts/" "02_DATA/VECTOR")

OUTPUT_DIR = PROJECT_DIR / "outputs"

rivers_path = DATA_DIR / "syr_rivers_3857.geojson"

water_bodies_path = DATA_DIR / "syr_lakes_4326.geojson"

admin0_path = DATA_DIR / "syr_admin0.geojson"

admin1_path = DATA_DIR / "syr_admin1.geojson"

aligned_hydrography_path = OUTPUT_DIR / "syr_hydrography_aligned_4326.gpkg"

output_path = PROJECT_DIR / "04_syria_vector_crs_alignment.html"

required_input_paths = {
    "River dataset": rivers_path,
    "Water-body dataset": water_bodies_path,
    "Country boundary dataset": admin0_path,
    "Governorate boundary dataset": admin1_path,
}

missing_input_paths = [
    path for path in required_input_paths.values() if not path.exists()
]

if missing_input_paths:
    raise FileNotFoundError(
        "One or more required input files were not found:\n"
        + "\n".join(str(path) for path in missing_input_paths)
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"River dataset: {rivers_path}")
print(f"Water-body dataset: {water_bodies_path}")
print(f"Country boundary dataset: {admin0_path}")
print(f"Governorate boundary dataset: {admin1_path}")
print(f"Aligned GeoPackage: {aligned_hydrography_path}")
print(f"Interactive map: {output_path}")

In [ ]:
# 3
# Read the hydrography and administrative boundary datasets
# 水文データと行政界データを読み込む

rivers = gpd.read_file(rivers_path)

water_bodies = gpd.read_file(water_bodies_path)

admin0 = gpd.read_file(admin0_path)

admin1 = gpd.read_file(admin1_path)

print(f"River features: {len(rivers):,}")
print(f"Water-body features: {len(water_bodies):,}")
print(f"Country boundary features: {len(admin0):,}")
print(f"Governorate boundary features: {len(admin1):,}")

In [ ]:
# 4
# Validate the source coordinate reference systems
# 入力データの座標参照系を検証する

datasets = {
    "Rivers": rivers,
    "Water bodies": water_bodies,
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

expected_epsg_codes = {
    "Rivers": 3857,
    "Water bodies": 4326,
    "Country boundaries": 4326,
    "Governorate boundaries": 4326,
}

for dataset_name, dataset in datasets.items():

    if dataset.crs is None:
        raise ValueError(f"{dataset_name} has no defined CRS.")

    actual_epsg = dataset.crs.to_epsg()
    expected_epsg = expected_epsg_codes[dataset_name]

    if actual_epsg != expected_epsg:
        raise ValueError(
            f"{dataset_name} was expected to use "
            f"EPSG:{expected_epsg}, but its CRS is "
            f"{dataset.crs}."
        )

    print(f"{dataset_name} CRS: {dataset.crs}")

print("Source CRS validation: passed")

In [ ]:
# 5
# Validate the source river features
# CRS変換前の河川データを検証する

required_river_columns = {
    "geometry",
}

missing_river_columns = required_river_columns - set(rivers.columns)

if missing_river_columns:
    raise ValueError(
        "The river dataset is missing required columns: "
        f"{sorted(missing_river_columns)}"
    )

if rivers.empty:
    raise ValueError("The river dataset contains no features.")

if rivers.geometry.isna().any():
    raise ValueError("The river dataset contains missing geometries.")

if rivers.geometry.is_empty.any():
    raise ValueError("The river dataset contains empty geometries.")

if not rivers.geometry.is_valid.all():
    raise ValueError("The river dataset contains invalid geometries.")

allowed_river_geometry_types = {
    "LineString",
    "MultiLineString",
}

unexpected_river_geometry_types = (
    set(rivers.geom_type.unique()) - allowed_river_geometry_types
)

if unexpected_river_geometry_types:
    raise ValueError(
        "The river dataset contains unexpected geometry types: "
        f"{sorted(unexpected_river_geometry_types)}"
    )

print("River dataset validation: passed")

print(rivers.geom_type.value_counts())

print(
    "River bounds:",
    rivers.total_bounds,
)

In [ ]:
# 6
# Validate and inspect the mapped water-body features
# 水域データのジオメトリと分類内容を確認する

required_water_body_columns = {
    "geometry",
}

missing_water_body_columns = required_water_body_columns - set(water_bodies.columns)

if missing_water_body_columns:
    raise ValueError(
        "The water-body dataset is missing required columns: "
        f"{sorted(missing_water_body_columns)}"
    )

if water_bodies.empty:
    raise ValueError("The water-body dataset contains no features.")

if water_bodies.geometry.isna().any():
    raise ValueError("The water-body dataset contains missing geometries.")

if water_bodies.geometry.is_empty.any():
    raise ValueError("The water-body dataset contains empty geometries.")

if not water_bodies.geometry.is_valid.all():
    raise ValueError("The water-body dataset contains invalid geometries.")

allowed_water_body_geometry_types = {
    "Polygon",
    "MultiPolygon",
}

unexpected_water_body_geometry_types = (
    set(water_bodies.geom_type.unique()) - allowed_water_body_geometry_types
)

if unexpected_water_body_geometry_types:
    raise ValueError(
        "The water-body dataset contains unexpected "
        "geometry types: "
        f"{sorted(unexpected_water_body_geometry_types)}"
    )

print("Water-body dataset validation: passed")

print(water_bodies.geom_type.value_counts())

if "water" in water_bodies.columns:
    print("\nMapped water-body classifications:")

    print(water_bodies["water"].fillna("Unclassified").value_counts())

if "natural" in water_bodies.columns:
    print("\nNatural-feature classifications:")

    print(water_bodies["natural"].fillna("Unclassified").value_counts())

print(
    "\nWater-body bounds:",
    water_bodies.total_bounds,
)

In [ ]:
# 7
# Validate the administrative boundary datasets
# 地図表示に使用する行政界データを検証する

required_admin_columns = {
    "Country boundaries": {
        "adm0_name",
        "adm0_pcode",
        "geometry",
    },
    "Governorate boundaries": {
        "adm1_name",
        "adm1_pcode",
        "geometry",
    },
}

administrative_datasets = {
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

for dataset_name, dataset in administrative_datasets.items():

    missing_columns = required_admin_columns[dataset_name] - set(dataset.columns)

    if missing_columns:
        raise ValueError(
            f"{dataset_name} is missing required columns: " f"{sorted(missing_columns)}"
        )

    if dataset.empty:
        raise ValueError(f"{dataset_name} contains no features.")

    if dataset.geometry.isna().any():
        raise ValueError(f"{dataset_name} contains missing geometries.")

    if dataset.geometry.is_empty.any():
        raise ValueError(f"{dataset_name} contains empty geometries.")

    if not dataset.geometry.is_valid.all():
        raise ValueError(f"{dataset_name} contains invalid geometries.")

    print(f"{dataset_name}: " f"{len(dataset):,} valid features")

print(
    admin0[
        [
            "adm0_name",
            "adm0_pcode",
        ]
    ]
)

print(
    admin1[
        [
            "adm1_name",
            "adm1_pcode",
        ]
    ].sort_values("adm1_name")
)

In [ ]:
# 8
# Reproject the river dataset to the water-body CRS
# 河川データを水域データと同じ座標参照系へ変換する

TARGET_CRS = water_bodies.crs

source_river_count = len(rivers)

rivers_aligned = rivers.to_crs(TARGET_CRS)

print(f"Source river CRS: {rivers.crs}")

print(f"Target CRS: {TARGET_CRS}")

print(f"Aligned river CRS: {rivers_aligned.crs}")

print(f"Aligned river features: {len(rivers_aligned):,}")

In [ ]:
# 9
# Validate the reprojected river dataset
# CRS変換後の河川データを検証する

if rivers_aligned.crs != TARGET_CRS:
    raise ValueError("The reprojected river dataset does not use " "the target CRS.")

if len(rivers_aligned) != source_river_count:
    raise ValueError("The river feature count changed during " "reprojection.")

if rivers_aligned.geometry.isna().any():
    raise ValueError("The reprojected river dataset contains " "missing geometries.")

if rivers_aligned.geometry.is_empty.any():
    raise ValueError("The reprojected river dataset contains " "empty geometries.")

if not rivers_aligned.geometry.is_valid.all():
    raise ValueError("The reprojected river dataset contains " "invalid geometries.")

source_geometry_types = set(rivers.geom_type.unique())

aligned_geometry_types = set(rivers_aligned.geom_type.unique())

if aligned_geometry_types != source_geometry_types:
    raise ValueError("One or more river geometry types changed " "during reprojection.")

print("River reprojection validation: passed")

print(rivers_aligned.geom_type.value_counts())

print(
    "Aligned river bounds:",
    rivers_aligned.total_bounds,
)

In [ ]:
# 10
# Confirm that the hydrography layers now share one CRS
# 河川データと水域データの座標参照系が統一されたことを確認する

aligned_hydrography = {
    "Aligned rivers": rivers_aligned,
    "Water bodies": water_bodies,
}

hydrography_crs_values = {str(layer.crs) for layer in aligned_hydrography.values()}

if len(hydrography_crs_values) != 1:
    raise ValueError("The river and water-body datasets do not " "share the same CRS.")

print("Hydrography CRS alignment: passed")

for layer_name, layer in aligned_hydrography.items():

    print(f"{layer_name}: " f"{len(layer):,} features, {layer.crs}")

In [ ]:
# 11
# Save the full-resolution aligned hydrography datasets
# CRSを統一した全解像度の水文データを保存する

if aligned_hydrography_path.exists():
    aligned_hydrography_path.unlink()

rivers_aligned.to_file(
    aligned_hydrography_path,
    layer="rivers",
    driver="GPKG",
)

water_bodies.to_file(
    aligned_hydrography_path,
    layer="water_bodies",
    driver="GPKG",
    mode="a",
)

print(f"GeoPackage saved to: {aligned_hydrography_path}")

print(f"Saved river features: {len(rivers_aligned):,}")

print(f"Saved water-body features: {len(water_bodies):,}")

In [ ]:
# 12
# Read back and verify the saved GeoPackage layers
# 保存したGeoPackageを再読込し、CRSと地物数を確認する

saved_rivers = gpd.read_file(
    aligned_hydrography_path,
    layer="rivers",
)

saved_water_bodies = gpd.read_file(
    aligned_hydrography_path,
    layer="water_bodies",
)

saved_layers = {
    "Saved rivers": {
        "data": saved_rivers,
        "expected_count": len(rivers_aligned),
    },
    "Saved water bodies": {
        "data": saved_water_bodies,
        "expected_count": len(water_bodies),
    },
}

for layer_name, layer_details in saved_layers.items():

    saved_layer = layer_details["data"]
    expected_count = layer_details["expected_count"]

    if saved_layer.crs != TARGET_CRS:
        raise ValueError(f"{layer_name} does not retain " "the target CRS.")

    if len(saved_layer) != expected_count:
        raise ValueError(f"{layer_name} does not retain " "the expected feature count.")

    print(f"{layer_name}: " f"{len(saved_layer):,} features, " f"{saved_layer.crs}")

print("Saved GeoPackage verification: passed")

In [ ]:
# 13
# Create simplified hydrography copies for web display
# Web地図表示用に水文データの軽量化コピーを作成する

DISPLAY_PROCESSING_CRS = "EPSG:32637"

RIVER_SIMPLIFICATION_METRES = 75
WATER_BODY_SIMPLIFICATION_METRES = 75

rivers_display_projected = saved_rivers.to_crs(DISPLAY_PROCESSING_CRS).copy()

water_bodies_display_projected = saved_water_bodies.to_crs(
    DISPLAY_PROCESSING_CRS
).copy()

rivers_display_projected["geometry"] = rivers_display_projected.geometry.simplify(
    tolerance=RIVER_SIMPLIFICATION_METRES,
    preserve_topology=True,
)

water_bodies_display_projected["geometry"] = (
    water_bodies_display_projected.geometry.simplify(
        tolerance=WATER_BODY_SIMPLIFICATION_METRES,
        preserve_topology=True,
    )
)

rivers_display_projected = rivers_display_projected[
    rivers_display_projected.geometry.notna()
    & ~rivers_display_projected.geometry.is_empty
].copy()

water_bodies_display_projected = water_bodies_display_projected[
    water_bodies_display_projected.geometry.notna()
    & ~water_bodies_display_projected.geometry.is_empty
].copy()

invalid_river_count = int((~rivers_display_projected.geometry.is_valid).sum())

invalid_water_body_count = int(
    (~water_bodies_display_projected.geometry.is_valid).sum()
)

if invalid_river_count > 0:
    rivers_display_projected["geometry"] = (
        rivers_display_projected.geometry.make_valid()
    )

if invalid_water_body_count > 0:
    water_bodies_display_projected["geometry"] = (
        water_bodies_display_projected.geometry.make_valid()
    )

rivers_display = rivers_display_projected.to_crs(TARGET_CRS)

water_bodies_display = water_bodies_display_projected.to_crs(TARGET_CRS)

print("Invalid river geometries after simplification: " f"{invalid_river_count:,}")

print(
    "Invalid water-body geometries after simplification: "
    f"{invalid_water_body_count:,}"
)

print(f"River display features: {len(rivers_display):,}")

print("Water-body display features: " f"{len(water_bodies_display):,}")

In [ ]:
# 14
# Validate the simplified web-display layers
# 軽量化したWeb地図表示用レイヤーを検証する

display_layers = {
    "River display layer": rivers_display,
    "Water-body display layer": water_bodies_display,
}

for layer_name, layer in display_layers.items():

    if layer.crs != TARGET_CRS:
        raise ValueError(f"{layer_name} does not use " "the target CRS.")

    if layer.empty:
        raise ValueError(f"{layer_name} contains no features.")

    if layer.geometry.isna().any():
        raise ValueError(f"{layer_name} contains missing geometries.")

    if layer.geometry.is_empty.any():
        raise ValueError(f"{layer_name} contains empty geometries.")

    if not layer.geometry.is_valid.all():
        raise ValueError(f"{layer_name} contains invalid geometries " "after repair.")

    print(f"{layer_name}: " f"{len(layer):,} valid features, " f"{layer.crs}")

print("Web-display layer validation: passed")

In [ ]:
# 15
# Create the no-label base map and set the initial extent
# 地名表記のないベースマップを作成し、初期表示範囲を設定する

(
    country_min_x,
    country_min_y,
    country_max_x,
    country_max_y,
) = admin0.total_bounds

m = folium.Map(
    location=[
        35.0,
        38.5,
    ],
    zoom_start=6,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
)

folium.TileLayer(
    tiles=("https://{s}.basemaps.cartocdn.com/" "light_nolabels/{z}/{x}/{y}{r}.png"),
    attr=(
        "&copy; "
        '<a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap</a> contributors "
        "&copy; "
        '<a href="https://carto.com/attributions">'
        "CARTO</a>"
    ),
    name="CARTO Light — No Labels",
    control=True,
    show=True,
).add_to(m)

m.fit_bounds(
    [
        [
            country_min_y,
            country_min_x,
        ],
        [
            country_max_y,
            country_max_x,
        ],
    ],
    padding=[
        25,
        25,
    ],
)

print(
    "Initial map extent:",
    [
        country_min_x,
        country_min_y,
        country_max_x,
        country_max_y,
    ],
)

In [ ]:
# 16
# Add the mapped water-body layer
# 軽量化した水域レイヤーを地図へ追加する

water_body_display_columns = [
    column
    for column in [
        "name",
        "name:en",
        "water",
        "natural",
        "geometry",
    ]
    if column in water_bodies_display.columns
]

water_body_tooltip_fields = [
    column
    for column in [
        "name",
        "name:en",
        "water",
        "natural",
    ]
    if column in water_body_display_columns
]

water_body_tooltip_aliases = {
    "name": "Name:",
    "name:en": "English name:",
    "water": "Water classification:",
    "natural": "Natural classification:",
}

water_body_layer = folium.FeatureGroup(
    name="Mapped Water Bodies",
    show=True,
)

water_body_geojson_arguments = {
    "data": water_bodies_display[water_body_display_columns].to_json(),
    "name": "Mapped Water Bodies",
    "style_function": lambda feature: {
        "fillColor": "#66c2d7",
        "color": "#168aad",
        "weight": 0.8,
        "fillOpacity": 0.58,
    },
    "highlight_function": lambda feature: {
        "fillColor": "#90e0ef",
        "color": "#0077b6",
        "weight": 2,
        "fillOpacity": 0.78,
    },
}

if water_body_tooltip_fields:
    water_body_geojson_arguments["tooltip"] = folium.GeoJsonTooltip(
        fields=water_body_tooltip_fields,
        aliases=[
            water_body_tooltip_aliases[field] for field in water_body_tooltip_fields
        ],
        localize=True,
        sticky=False,
        labels=True,
    )

folium.GeoJson(**water_body_geojson_arguments).add_to(water_body_layer)

water_body_layer.add_to(m)

print("Mapped water-body layer added: " f"{len(water_bodies_display):,} features")

In [ ]:
# 17
# Add the aligned river layer
# CRSを統一した河川レイヤーを地図へ追加する

river_display_columns = [
    column
    for column in [
        "name",
        "name:en",
        "waterway",
        "geometry",
    ]
    if column in rivers_display.columns
]

river_tooltip_fields = [
    column
    for column in [
        "name",
        "name:en",
        "waterway",
    ]
    if column in river_display_columns
]

river_tooltip_aliases = {
    "name": "Name:",
    "name:en": "English name:",
    "waterway": "Waterway type:",
}

river_layer = folium.FeatureGroup(
    name="Aligned Rivers",
    show=True,
)

river_geojson_arguments = {
    "data": rivers_display[river_display_columns].to_json(),
    "name": "Aligned Rivers",
    "style_function": lambda feature: {
        "color": "#023e8a",
        "weight": 1.25,
        "opacity": 0.82,
    },
    "highlight_function": lambda feature: {
        "color": "#00b4d8",
        "weight": 3,
        "opacity": 1.0,
    },
}

if river_tooltip_fields:
    river_geojson_arguments["tooltip"] = folium.GeoJsonTooltip(
        fields=river_tooltip_fields,
        aliases=[river_tooltip_aliases[field] for field in river_tooltip_fields],
        localize=True,
        sticky=False,
        labels=True,
    )

folium.GeoJson(**river_geojson_arguments).add_to(river_layer)

river_layer.add_to(m)

print("Aligned river layer added: " f"{len(rivers_display):,} features")

In [ ]:
# 18
# Add the country and governorate boundaries
# シリア国境および県境レイヤーを追加する

country_boundary_layer = folium.FeatureGroup(
    name="Syria Boundary",
    show=True,
)

folium.GeoJson(
    data=admin0[
        [
            "adm0_name",
            "adm0_pcode",
            "geometry",
        ]
    ].to_json(),
    name="Syria Boundary",
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "#202020",
        "weight": 2.7,
        "fillOpacity": 0.0,
        "opacity": 0.95,
    },
    highlight_function=lambda feature: {
        "color": "#111111",
        "weight": 3.2,
        "fillOpacity": 0.02,
        "opacity": 1.0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm0_name",
            "adm0_pcode",
        ],
        aliases=[
            "Country:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(country_boundary_layer)

country_boundary_layer.add_to(m)

governorate_boundary_layer = folium.FeatureGroup(
    name="Governorate Boundaries",
    show=True,
)

folium.GeoJson(
    data=admin1[
        [
            "adm1_name",
            "adm1_pcode",
            "geometry",
        ]
    ].to_json(),
    name="Governorate Boundaries",
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "#555555",
        "weight": 1.0,
        "fillOpacity": 0.0,
        "opacity": 0.8,
    },
    highlight_function=lambda feature: {
        "color": "#111111",
        "weight": 2.2,
        "fillOpacity": 0.03,
        "opacity": 1.0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "Governorate:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(governorate_boundary_layer)

governorate_boundary_layer.add_to(m)

print("Administrative boundary layers added")

In [ ]:
# 19
# Add governorate and neighbouring-country labels
# 県名および周辺国名を追加する

governorate_label_layer = folium.FeatureGroup(
    name="Governorate Labels",
    show=True,
)

admin1_label_points = admin1.copy()

has_label_coordinates = {
    "center_lat",
    "center_lon",
}.issubset(admin1_label_points.columns) and admin1_label_points[
    [
        "center_lat",
        "center_lon",
    ]
].notna().all().all()

if has_label_coordinates:
    admin1_label_points["label_latitude"] = admin1_label_points["center_lat"]

    admin1_label_points["label_longitude"] = admin1_label_points["center_lon"]

else:
    representative_points = admin1_label_points.to_crs(
        DISPLAY_PROCESSING_CRS
    ).geometry.representative_point()

    representative_points = gpd.GeoSeries(
        representative_points,
        crs=DISPLAY_PROCESSING_CRS,
    ).to_crs(TARGET_CRS)

    admin1_label_points["label_latitude"] = representative_points.y

    admin1_label_points["label_longitude"] = representative_points.x

for _, governorate in admin1_label_points.iterrows():

    folium.Marker(
        location=[
            governorate["label_latitude"],
            governorate["label_longitude"],
        ],
        icon=folium.DivIcon(
            icon_size=(
                130,
                24,
            ),
            icon_anchor=(
                65,
                12,
            ),
            html=f"""
            <div style="
                font-size: 12px;
                font-weight: 700;
                color: #202020;
                text-align: center;
                white-space: nowrap;
                text-shadow:
                    -1px -1px 0 #ffffff,
                     1px -1px 0 #ffffff,
                    -1px  1px 0 #ffffff,
                     1px  1px 0 #ffffff;
            ">
                {governorate["adm1_name"]}
            </div>
            """,
        ),
    ).add_to(governorate_label_layer)

governorate_label_layer.add_to(m)

neighbour_label_layer = folium.FeatureGroup(
    name="Neighbour Labels",
    show=True,
)

neighbour_labels = {
    "TÜRKİYE": [
        37.75,
        38.1,
    ],
    "IRAQ": [
        34.5,
        42.75,
    ],
    "LEBANON": [
        33.8,
        35.5,
    ],
    "JORDAN": [
        31.85,
        37.2,
    ],
}

for country_name, coordinates in neighbour_labels.items():

    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            icon_size=(
                150,
                30,
            ),
            icon_anchor=(
                75,
                15,
            ),
            html=f"""
            <div style="
                font-size: 18px;
                font-weight: 700;
                color: #666666;
                text-align: center;
                white-space: nowrap;
                text-shadow:
                    -1px -1px 0 #ffffff,
                     1px -1px 0 #ffffff,
                    -1px  1px 0 #ffffff,
                     1px  1px 0 #ffffff;
            ">
                {country_name}
            </div>
            """,
        ),
    ).add_to(neighbour_label_layer)

neighbour_label_layer.add_to(m)

print("Governorate and neighbouring-country labels added")

In [ ]:
# 20
# Add the map information and source panel
# 地図の説明、CRS変換方法および出典を追加する

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 440px;
    min-height: 225px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="font-size: 16px;">
        Syria
    </b>
    <br>

    <span style="
        color: #0077b6;
        font-weight: bold;
    ">
        Hydrography CRS Alignment
    </span>

    <small style="
        display: block;
        margin-top: 7px;
        line-height: 1.35;
        color: #333333;
    ">
        River features were reprojected from
        Web Mercator (EPSG:3857) to
        WGS 84 (EPSG:4326) and aligned with the
        mapped water-body dataset.
        The GeoPackage retains the full-resolution
        geometries, while the interactive map uses
        simplified display copies.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        font-size: 11px;
        line-height: 1.35;
        color: #555555;
        border-top: 1px solid #aaaaaa;
    ">
        Hydrography source:
        <a
            href="https://www.openstreetmap.org/copyright"
            target="_blank"
            style="
                color: #0077b6;
                text-decoration: none;
                font-weight: bold;
            "
        >
            OpenStreetMap contributors
        </a><br>

        Boundary source:
        <b>HDX OCHA</b><br>

        River features:
        <b>{len(saved_rivers):,}</b><br>

        Mapped water-body features:
        <b>{len(saved_water_bodies):,}</b><br>

        Source river CRS:
        <b>EPSG:3857</b><br>

        Target CRS:
        <b>EPSG:4326</b><br>

        Method:
        CRS Validation / Reprojection /
        GeoPackage Export / Web Simplification
    </div>
</div>
"""

m.get_root().html.add_child(Element(information_panel_html))

In [ ]:
# 21
# Add the hydrography legend
# 水文データの凡例を追加する

legend_html = """
<div style="
    position: fixed;
    right: 35px;
    bottom: 25px;
    width: 285px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 13px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 12px rgba(0, 0, 0, 0.22);
">
    <b style="font-size: 15px;">
        Hydrography
    </b>

    <div style="
        display: flex;
        align-items: center;
        margin-top: 12px;
    ">
        <span style="
            display: inline-block;
            width: 42px;
            height: 17px;
            margin-right: 10px;
            background-color: rgba(102, 194, 215, 0.58);
            border: 1px solid #168aad;
        "></span>
        Mapped water body
    </div>

    <div style="
        display: flex;
        align-items: center;
        margin-top: 10px;
    ">
        <span style="
            display: inline-block;
            width: 42px;
            height: 0;
            margin-right: 10px;
            border-top: 3px solid #023e8a;
        "></span>
        Aligned river feature
    </div>

    <div style="
        margin-top: 12px;
        padding-top: 8px;
        border-top: 1px solid #aaaaaa;
        font-size: 11px;
        line-height: 1.4;
        color: #555555;
    ">
        Source river CRS: EPSG:3857<br>
        Target CRS: EPSG:4326<br>
        River display simplification: 75 m<br>
        Water-body display simplification: 75 m
    </div>
</div>
"""

m.get_root().html.add_child(Element(legend_html))

In [ ]:
# 22
# Add the layer control
# 地図レイヤーの表示と非表示を切り替える機能を追加する

folium.LayerControl(
    position="topright",
    collapsed=False,
).add_to(m)

In [ ]:
# 23
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(output_path)

print(f"Map saved to: {output_path}")

m